In [1]:
import requests
import pandas as pd
import json

import random
import time

HEADERS = {"User-Agent": "weather-data-acquisition-project"}

In [4]:
def http_get(url, params=None, headers=HEADERS, attempts=5):
    for k in range(attempts):
        try:
            r = requests.get(
                url, params=params, headers=headers, timeout=10
            )
        except requests.RequestException as exc:
            status, retry_after = repr(exc), 0
        else:
            # 1. Return the JSON if the request succeeds
            if r.status_code == 200:
                return r.json()

            # 2. Raise an error for non-retryable status codes
            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()

            # 3. Read the status and Retry-After header
            status = r.status_code
            value = r.headers.get("Retry-After", "0")
            retry_after = float(value) if value.isdecimal() else 0

        # 4. Wait before retrying
        if k + 1 < attempts:
            wait = retry_after if retry_after > 0 else random.uniform(0, 2**k)
            print(f"  {status} on attempt {k + 1}, sleeping {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(f"gave up after {attempts} attempts: {url}")

In [16]:
import hashlib
from pathlib import Path
requests_made = 0
CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)
FIXTURES = Path("fixtures")
POLITE_DELAY = 1.0

def cache_key(url, params=None):
    """A short, stable name for one (url, params) pair.
    """
    payload = url + "|" + json.dumps(params or {}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

def cached_get(url, params=None):
    path = CACHE / f"{cache_key(url, params)}.json"
    if path.exists():
        return json.loads(path.read_text())

    global requests_made
    requests_made += 1
    body = http_get(url, params)
    path.write_text(json.dumps(body))
    time.sleep(POLITE_DELAY)
    return body

In [11]:
# Define the time period for the weather data
start_date = "2024-10-01"
end_date = "2026-09-30"

In [9]:
# Define the locations and their geographic coordinates
locations = {
    "Grenoble": {
        "latitude": 45.18,
        "longitude": 5.72
    },
    "Chamrousse": {
        "latitude": 45.13,
        "longitude": 5.88
    },
    "Voiron": {
        "latitude": 45.36,
        "longitude": 5.59
    }
}

In [6]:
# Define the Open-Meteo Historical Weather API endpoint
url = "https://archive-api.open-meteo.com/v1/archive"

### API AUTHENTICATION  
The Open-Meteo Historical Weather API does not require an API key for non-commercial use.

Therefore, our acquisition script can access the historical weather data directly through HTTP GET requests without authentication.

No credentials or personal access tokens are stored in our notebook.

### API Rate Limits

Open-Meteo applies usage limits to its free, non-commercial API.

Our project requests historical weather data for three locations: Grenoble, Chamrousse, and Voiron.

To avoid sending unnecessary requests, we implemented:

- **Caching:** previously downloaded responses are stored locally and reused.
- **Polite delay:** a one-second delay between new requests.
- **Retry mechanism:** temporary HTTP errors (such as 429 or 503) are handled with retries and exponential backoff.

These mechanisms reduce server load and make our data acquisition more reliable.

### API Response Structure

The Open-Meteo Historical Weather API returns data in JSON format.

The response contains:

- **latitude / longitude:** geographic coordinates of the requested location.
- **hourly:** hourly weather observations.
- **time:** timestamps corresponding to each observation.
- **temperature_2m:** air temperature in °C.
- **relative_humidity_2m:** relative humidity in %.
- **precipitation:** precipitation in mm.
- **wind_speed_10m:** wind speed in km/h.
- **snow_depth:** snow depth in meters.

The hourly variables are represented as lists. Values at the same index correspond to the same timestamp.

We convert these lists into a pandas DataFrame for further processing and analysis.

In [7]:
# Define the hourly weather variables to retrieve
weather_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "snow_depth"
]

In [12]:
# Define the parameters for a first API request for Grenoble
params = {
    "latitude": locations["Grenoble"]["latitude"],
    "longitude": locations["Grenoble"]["longitude"],
    "start_date": start_date,
    "end_date": end_date,
    "hourly": weather_variables
}

In [17]:
# Send a GET request using the cache
data = cached_get(url, params=params)

# Display the first keys of the JSON response
print(data.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])


In [31]:
# Inspect the hourly weather data returned by the API
print(data["hourly"].keys())

dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'snow_depth'])


In [32]:
# Display the first five timestamps and temperatures
print(data["hourly"]["time"][:5])
print(data["hourly"]["temperature_2m"][:5])

['2024-10-01T00:00', '2024-10-01T01:00', '2024-10-01T02:00', '2024-10-01T03:00', '2024-10-01T04:00']
[16.1, 15.8, 15.8, 15.7, 15.7]


In [33]:
# Convert the hourly JSON data into a Pandas DataFrame
df = pd.DataFrame(data["hourly"])

# Display the first five rows
df.head()

,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,snow_depth
0,2024-10-01T00:00,16.1,82,0.0,5.4,0.0
1,2024-10-01T01:00,15.8,85,0.0,4.9,0.0
2,2024-10-01T02:00,15.8,84,0.0,4.7,0.0
3,2024-10-01T03:00,15.7,86,0.0,5.0,0.0
4,2024-10-01T04:00,15.7,86,0.0,4.9,0.0


In [34]:
# Display the number of rows and columns in the DataFrame
print("Shape:", df.shape)

Shape: (17520, 6)


In [35]:
# Check the first and last timestamps in the dataset
print("First timestamp:", df["time"].min())
print("Last timestamp:", df["time"].max())

First timestamp: 2024-10-01T00:00
Last timestamp: 2026-09-30T23:00


### Pagination

The Open-Meteo Historical Weather API does not require pagination for our requests.

Each request returns the hourly data for one location.

We use a loop to collect data from three locations, not to paginate.

In [18]:
# Create an empty list to store the data from all locations
all_data = []

# Loop through each location
for name, coordinates in locations.items():

    # Define the parameters of the API request for the current location
    params = {
        "latitude": coordinates["latitude"],
        "longitude": coordinates["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m",
            "snow_depth"
        ]
    }

    data = cached_get(url, params=params)
    # Convert the hourly weather data into a Pandas DataFrame
    df_location = pd.DataFrame(data["hourly"])

    # Add the location name to the DataFrame
    df_location["location"] = name

    # Add this DataFrame to the list
    all_data.append(df_location)

    # Display the location and the shape of its DataFrame
    print(name, df_location.shape)

Grenoble (17520, 7)
Chamrousse (17520, 7)
Voiron (17520, 7)


In [19]:
print("Requests made:", requests_made)

# Repeat the same request twice
first = cached_get(url, params=params)
second = cached_get(url, params=params)

print("Identical:", first == second)
print("Requests made:", requests_made)
print("Cache files:", len(list(CACHE.glob("*.json"))))

Requests made: 3
Identical: True
Requests made: 3
Cache files: 3


In [20]:
# Combine the data from all locations into one DataFrame
df_all = pd.concat(all_data, ignore_index=True)

# Display the shape and the first rows of the final DataFrame
print("Shape:", df_all.shape)
df_all.head()

Shape: (52560, 7)


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,snow_depth,location
0,2024-10-01T00:00,16.1,82,0.0,5.4,0.0,Grenoble
1,2024-10-01T01:00,15.8,85,0.0,4.9,0.0,Grenoble
2,2024-10-01T02:00,15.8,84,0.0,4.7,0.0,Grenoble
3,2024-10-01T03:00,15.7,86,0.0,5.0,0.0,Grenoble
4,2024-10-01T04:00,15.7,86,0.0,4.9,0.0,Grenoble


In [21]:
from pathlib import Path

output_path = Path("weather_data.parquet")

if output_path.exists():
    print("File already exists. Skipping save.")
else:
    df_all.to_parquet(output_path, index=False)
    print("File saved successfully.")

File already exists. Skipping save.


In [39]:
# Create a log describing the data acquisition
acquisition_log = {
    "source": "Open-Meteo",
    "endpoint": url,

    # Document the scope revision made for M1
    "scope_revision": (
        "The registered scope referred to Grenoble and nearby weather stations. "
        "For M1, the scope is revised to Grenoble and nearby geographic locations "
        "queried by latitude and longitude through the Open-Meteo Historical Weather API."
    ),

    "start_date": start_date,
    "end_date": end_date,
    "locations": locations,
    "variables": weather_variables,
    "parquet_file": "weather_data.parquet",
    "shape": list(df_all.shape)
}


In [22]:
from pathlib import Path

# Preserve the original acquisition log used in M2 and M3.
log_path = Path("acquisition_log.json")

if log_path.exists():
    print("Acquisition log already exists. Skipping save.")
else:
    with open(log_path, "w") as f:
        json.dump(log, f, indent=2)
    print("Acquisition log saved successfully.")

Acquisition log already exists. Skipping save.


In [23]:
# Read the acquisition log to check its content
with open("acquisition_log.json", "r") as file:
    log = json.load(file)

print(log)

{'source': 'Open-Meteo', 'endpoint': 'https://archive-api.open-meteo.com/v1/archive', 'scope_revision': 'The registered scope referred to Grenoble and nearby weather stations. For M1, the scope is revised to Grenoble and nearby geographic locations queried by latitude and longitude through the Open-Meteo Historical Weather API.', 'start_date': '2024-10-01', 'end_date': '2026-09-30', 'locations': {'Grenoble': {'latitude': 45.18, 'longitude': 5.72}, 'Chamrousse': {'latitude': 45.13, 'longitude': 5.88}, 'Voiron': {'latitude': 45.36, 'longitude': 5.59}}, 'variables': ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m', 'snow_depth'], 'parquet_file': 'weather_data.parquet', 'shape': [52560, 7]}
